**- Setup**

In [0]:
from pyspark.sql.functions import (
    col, current_date, current_timestamp, lit, when, 
    coalesce, hash, concat_ws, sha2
)
from pyspark.sql.types import StringType
from delta.tables import DeltaTable

# Widget defaults
dbutils.widgets.text("storage_account", "retailprojec1data")
dbutils.widgets.text("bronze_container", "bronze")
dbutils.widgets.text("silver_container", "silver")

storage_account = dbutils.widgets.get("storage_account")
bronze_container = dbutils.widgets.get("bronze_container")
silver_container = dbutils.widgets.get("silver_container")

# Paths
bronze_products_path = f"abfss://{bronze_container}@{storage_account}.dfs.core.windows.net/products_delta/"
silver_dim_path = f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net/dim_products/"

print(f"Bronze:  {bronze_products_path}")
print(f"Silver:  {silver_dim_path}")

**- Reading bronze**

In [0]:
bronze_products = spark.read.format("delta").load(bronze_products_path)
print(f"Bronze products: {bronze_products.count()}")
bronze_products.printSchema()
bronze_products.show(5, truncate=False)

**-Defining tracked columns**

In [0]:
# Columns that trigger SCD2 versioning when changed
TRACKED_COLUMNS = [
    "product_name",
    "category",
    "retail_price",
    "cost"
]

# Prepare incoming with row_hash for change detection
incoming = (bronze_products
    .select(
        "sku",
        "product_name",
        "category",
        "retail_price",
        "cost",
        "created_date"
    )
    .withColumn(
        "row_hash",
        hash(concat_ws("||", *[coalesce(col(c), lit("NULL")).cast("string") for c in TRACKED_COLUMNS]))
    )
)

print(f"Incoming rows: {incoming.count()}")
incoming.show(3, truncate=False)

**-Initializing silver if empty**

In [0]:
silver_exists = False

try:
    test_df = spark.read.format("delta").load(silver_dim_path)
    row_count = test_df.count()
    
    if row_count > 0:
        silver_exists = True
        print(f"✅ Silver dim_products EXISTS with {row_count} rows")
    else:
        silver_exists = False
        print(f"⚠️ Silver location exists but is EMPTY — treating as new")
except Exception as e:
    silver_exists = False
    print(f"⚠️ Cannot read Silver dim_products: {type(e).__name__}")

print(f"\nsilver_exists = {silver_exists}")

if not silver_exists:
    print("\n>>> ENTERING INITIALIZATION BLOCK <<<")
    from pyspark.sql.functions import to_date
    initial = (incoming
    .withColumn("effective_from", to_date(col("created_date")))
    .withColumn(
        "product_sk",
        sha2(
            concat_ws("||", 
                col("sku").cast("string"),
                col("effective_from").cast("string"),
                col("row_hash").cast("string")
            ),
            256
        )
    )
    .withColumn("effective_to", lit(None).cast("date"))
    .withColumn("is_current", lit(True))
    .withColumn("silver_ingestion_ts", current_timestamp())
)
    final_columns = (
        ["product_sk", "sku"]
        + TRACKED_COLUMNS
        + ["effective_from", "effective_to", "is_current", "row_hash", "silver_ingestion_ts"]
    )
    initial = initial.select(*final_columns)
    
    (initial.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(silver_dim_path))
    
    initial_count = spark.read.format("delta").load(silver_dim_path).count()
    print(f"\n✅ Silver dim_products initialized with {initial_count} rows")
    initial.select("sku", "product_name", "product_sk").show(3, truncate=False)
else:
    print("\n>>> SKIPPING INITIALIZATION — Silver exists with data <<<")

**-Applying SCD2 Merge**

In [0]:
if silver_exists:
    silver_full = spark.read.format("delta").load(silver_dim_path)
    silver_current = silver_full.filter(col("is_current") == True)
    
    # Detect changed products
    changes = (incoming.alias("new")
        .join(silver_current.alias("old"), "sku")
        .filter(col("new.row_hash") != col("old.row_hash"))
        .select(col("new.sku"))
    )
    
    changed_skus = [row.sku for row in changes.collect()]
    print(f"Products with changes: {len(changed_skus)}")
    
    # Detect new products
    new_products_df = (incoming.alias("new")
        .join(silver_current.alias("old"), "sku", "left_anti")
    )
    
    new_count = new_products_df.count()
    print(f"Brand new products: {new_count}")
    
    # Close out old rows for changed products
    if len(changed_skus) > 0:
        silver_table = DeltaTable.forPath(spark, silver_dim_path)
        silver_table.update(
            condition = (col("sku").isin(changed_skus)) & (col("is_current") == True),
            set = {
                "effective_to": "current_date()",
                "is_current": "false"
            }
        )
        print(f"✅ Closed out {len(changed_skus)} old SCD2 rows")
    
    # Insert new SCD2 rows
    all_new_skus = changed_skus + [row.sku for row in new_products_df.select("sku").collect()]
    
    if len(all_new_skus) > 0:
        rows_to_insert = incoming.filter(col("sku").isin(all_new_skus))
        
        rows_with_sk = (rows_to_insert
            .withColumn("effective_from", current_date())
            .withColumn(
                "product_sk",
                sha2(
                    concat_ws("||", 
                        col("sku").cast("string"),
                        col("effective_from").cast("string"),
                        col("row_hash").cast("string")
                    ),
                    256
                )
            )
            .withColumn("effective_to", lit(None).cast("date"))
            .withColumn("is_current", lit(True))
            .withColumn("silver_ingestion_ts", current_timestamp())
        )
        
        final_columns = (
            ["product_sk", "sku"]
            + TRACKED_COLUMNS
            + ["effective_from", "effective_to", "is_current", "row_hash", "silver_ingestion_ts"]
        )
        rows_with_sk = rows_with_sk.select(*final_columns)
        
        rows_with_sk.write.format("delta").mode("append").save(silver_dim_path)
        print(f"✅ Inserted {rows_with_sk.count()} new SCD2 rows")
    else:
        print("No new rows to insert")

**-Verifying**

In [0]:
silver_dim = spark.read.format("delta").load(silver_dim_path)

print("=" * 60)
print("dim_products SCD2 Summary")
print("=" * 60)
print(f"Total rows:          {silver_dim.count()}")
print(f"Current rows:        {silver_dim.filter(col('is_current') == True).count()}")
print(f"Historical rows:     {silver_dim.filter(col('is_current') == False).count()}")
print(f"Unique SKUs:         {silver_dim.select('sku').distinct().count()}")

print("\nSample:")
(silver_dim
    .orderBy("sku", "effective_from")
    .select("product_sk", "sku", "product_name", "category", "retail_price", "is_current")
    .show(10, truncate=False))

print("\nSchema:")
silver_dim.printSchema()

**-Exit**

In [0]:
total = silver_dim.count()
current = silver_dim.filter(col("is_current") == True).count()
historical = silver_dim.filter(col("is_current") == False).count()

dbutils.notebook.exit(f"total_{total}_current_{current}_historical_{historical}")